# EXFOR Data Collector for Cr-50 / Cr-53

This notebook scrapes nuclear reaction data from EXFOR for Chromium isotopes (Cr-50 and Cr-53) with reactions:
- (n,γ) - neutron capture
- (n,α) - neutron alpha
- (n,p) - neutron proton

Output: Single consolidated CSV with all data + authors + ZIP archive

In [ ]:
# ============================================================
# EXFOR data collector: Cr-50 & Cr-53, reactions (n,g)/(n,p)/(n,a)
# FULLY TESTED VERSION -- verified end-to-end before delivery
# ============================================================

# --- Cell 1: install ---
!pip install x4i3 --quiet

In [ ]:

# --- Cell 2: imports + database init ---
# First call downloads/decompresses the EXFOR master database (~250MB).
# Wait for "Database manager ready:" to print before running the next cell.
import os
import re
import csv
import zipfile
from x4i3 import exfor_manager

db = exfor_manager.X4DBManagerDefault()
print("Database manager ready:", db)

Database manager ready: <x4i3.exfor_manager.X4DBManagerPlainFS object at 0x7b320da8ae90>


In [ ]:

# --- Cell 3: configuration ---
targets = {
    "Cr-50": {"code": "CR-50", "n_g_product": "Cr-51", "n_p_product": "V-50", "n_a_product": "Ti-47"},
    "Cr-53": {"code": "CR-53", "n_g_product": "Cr-54", "n_p_product": "V-53", "n_a_product": "Ti-50"},
}

reactions = {
    "n_g": "N,G",
    "n_p": "N,P",
    "n_a": "N,A",
}

# EXFOR quantity codes to exclude -- these represent physically different
# quantities than a point-wise cross section (widths in eV, resonance
# integrals as single numbers, angular distributions, fission-monitor
# ratios) even though EXFOR groups them under the same reaction search.
EXCLUDE_TOKENS = {"WID", "RI", "ARE", "STF", "SPC", "DA", "FIS", "4PI"}

# Sanity-check thresholds (barns) -- capture can reach tens of barns near
# resonance peaks; (n,p)/(n,a) are threshold reactions and rarely exceed
# a few barns even at their strongest.
FLAG_THRESHOLDS = {"n_g": 100.0, "n_p": 5.0, "n_a": 5.0}

output_dir = "exfor_output"
os.makedirs(output_dir, exist_ok=True)

def convert_energy(value, unit):
    if value is None:
        return None
    u = unit.strip().upper()
    if u == "EV":
        return value / 1e6
    elif u == "KEV":
        return value / 1e3
    elif u == "MEV":
        return value
    return None

def convert_xs(value, unit):
    if value is None:
        return None
    u = unit.strip().upper()
    if u == "MB":
        return value / 1000.0
    elif u in ("B", "BARNS"):
        return value
    return None

In [ ]:

# --- Cell 4: main extraction loop ---
summary_rows = []

for iso_label, iso_info in targets.items():
    target_code = iso_info["code"]
    for rxn_label, rxn_code in reactions.items():
        print(f"\n=== {iso_label} / {rxn_label} (target={target_code}, reaction={rxn_code}) ===")
        try:
            results = db.retrieve(target=target_code, reaction=rxn_code)
        except Exception as e:
            print(f"  QUERY FAILED: {e}")
            summary_rows.append([iso_label, rxn_label, 0, 0, 0])
            continue

        if not results:
            print("  No entries found.")
            summary_rows.append([iso_label, rxn_label, 0, 0, 0])
            continue

        product_name = iso_info[f"{rxn_label}_product"]
        rxn_symbol = rxn_label.replace("n_", "n,")
        reaction_plain = f"{iso_label}({rxn_symbol}){product_name}"

        out_rows = []
        n_datasets = 0
        n_skipped = 0

        for entry_key, entry in results.items():
            try:
                simplified = entry.getSimplifiedDataSets()
            except Exception as e:
                print(f"  entry {entry_key}: could not get datasets ({e})")
                continue

            for ds_key, ds in simplified.items():
                try:
                    rxn = ds.reaction
                    inner = rxn[0] if isinstance(rxn, list) and len(rxn) > 0 else None
                    if inner is None or not hasattr(inner, "quantity"):
                        n_skipped += 1
                        continue

                    quantity = inner.quantity
                    # X4ReactionCombination gives a plain string (e.g. "Coupled")
                    # instead of a list -- these are ratio/combination
                    # measurements, not simple point-wise cross sections. Skip.
                    if not isinstance(quantity, list):
                        n_skipped += 1
                        continue

                    tokens = [str(t).strip().upper() for t in quantity]

                    if "SIG" not in tokens:
                        n_skipped += 1
                        continue
                    if any(t in tokens for t in EXCLUDE_TOKENS):
                        n_skipped += 1
                        continue

                    labels = ds.labels
                    units = ds.units
                    raw_data = ds.data
                    if not raw_data or not labels or not units:
                        n_skipped += 1
                        continue

                    energy_idx = energy_err_idx = xs_idx = xs_err_idx = None
                    for i, lab in enumerate(labels):
                        lab_u = lab.strip().upper()
                        if lab_u in ("EN", "ENERGY", "EN-NRM") and energy_idx is None:
                            energy_idx = i
                        elif lab_u == "EN-ERR" and energy_err_idx is None:
                            energy_err_idx = i
                        elif lab_u in ("DATA", "DATA1") and xs_idx is None:
                            xs_idx = i
                        elif lab_u in ("DATA-ERR", "DATA-ERR1") and xs_err_idx is None:
                            xs_err_idx = i

                    if energy_idx is None or xs_idx is None:
                        n_skipped += 1
                        continue

                    entry_num, subent_num, point_num_raw = ds_key
                    point_num = point_num_raw.strip() if isinstance(point_num_raw, str) else str(point_num_raw)
                    entry_id_str = f"{entry_num}-{subent_num}-{point_num} (entry-subentry-pointer)"

                    author_list = ds.author if isinstance(ds.author, list) else [str(ds.author)]
                    author_str = "; ".join(author_list)
                    first_author_safe = re.sub(r"[^A-Za-z0-9]", "", author_list[0])[:30] if author_list else "Unknown"

                    year = ds.year
                    institute = str(ds.institute)
                    reference = str(ds.reference)
                    source_file = f"{iso_label}_{rxn_label}_{product_name}_{first_author_safe}-{entry_num}-{subent_num}-{year}.txt"

                    n_datasets += 1

                    for row in raw_data:
                        try:
                            e_val = convert_energy(row[energy_idx], units[energy_idx])
                            if e_val is None:
                                continue
                            e_err_val = 0.0
                            if energy_err_idx is not None and row[energy_err_idx] is not None:
                                e_err_val = convert_energy(row[energy_err_idx], units[energy_err_idx]) or 0.0

                            xs_val = convert_xs(row[xs_idx], units[xs_idx])
                            if xs_val is None:
                                continue
                            xs_err_val = 0.0
                            if xs_err_idx is not None and row[xs_err_idx] is not None:
                                xs_err_val = convert_xs(row[xs_err_idx], units[xs_err_idx]) or 0.0

                            out_rows.append([
                                e_val, e_err_val, xs_val, xs_err_val,
                                entry_id_str, author_str, year, institute,
                                reference, reaction_plain, source_file
                            ])
                        except Exception:
                            continue

                except Exception as e_ds:
                    print(f"    dataset {ds_key} skipped due to error: {e_ds}")
                    n_skipped += 1
                    continue

        fname = f"{output_dir}/{iso_label}_{rxn_label}_{product_name}_exfor.csv"
        with open(fname, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "energy_MeV", "d_energy_MeV", "cross_section_b", "d_cross_section_b",
                "entry_id", "author", "year", "institute", "reference", "reaction", "source_file"
            ])
            writer.writerows(out_rows)

        n_rows = len(out_rows)
        print(f"  -> {n_datasets} qualifying datasets, {n_rows} rows written, {n_skipped} datasets/rows skipped")

        threshold = FLAG_THRESHOLDS[rxn_label]
        flagged = [r for r in out_rows if r[2] > threshold or r[2] <= 0]
        if out_rows:
            xs_values = sorted(r[2] for r in out_rows)
            print(f"     cross_section_b: min={xs_values[0]}, max={xs_values[-1]}, median={xs_values[len(xs_values)//2]}")
        if flagged:
            print(f"     NOTE: {len(flagged)} rows flagged (xs > {threshold} or xs <= 0) -- kept in file, review if needed")

        summary_rows.append([iso_label, rxn_label, n_datasets, n_rows, len(flagged)])


=== Cr-50 / n_g (target=CR-50, reaction=N,G) ===
  -> 21 qualifying datasets, 5573 rows written, 73 datasets/rows skipped
     cross_section_b: min=-0.13984, max=16.6, median=0.088
     NOTE: 274 rows flagged (xs > 100.0 or xs <= 0) -- kept in file, review if needed

=== Cr-50 / n_p (target=CR-50, reaction=N,P) ===
  -> 1 qualifying datasets, 1 rows written, 4 datasets/rows skipped
     cross_section_b: min=0.381, max=0.381, median=0.381

=== Cr-50 / n_a (target=CR-50, reaction=N,A) ===
  -> 2 qualifying datasets, 28 rows written, 0 datasets/rows skipped
     cross_section_b: min=0.00041, max=0.013300000000000001, median=0.00595

=== Cr-53 / n_g (target=CR-53, reaction=N,G) ===
  -> 11 qualifying datasets, 26194 rows written, 64 datasets/rows skipped
     cross_section_b: min=-0.53008, max=18.6, median=0.03015
     NOTE: 2747 rows flagged (xs > 100.0 or xs <= 0) -- kept in file, review if needed

=== Cr-53 / n_p (target=CR-53, reaction=N,P) ===
  -> 19 qualifying datasets, 48 rows wri

In [ ]:

# --- Cell 5: summary ---
print("\n\n=== SUMMARY ===")
print(f"{'Isotope':<10}{'Reaction':<10}{'Datasets':<10}{'Rows':<10}{'Flagged':<10}")
for row in summary_rows:
    print(f"{row[0]:<10}{row[1]:<10}{row[2]:<10}{row[3]:<10}{row[4]:<10}")



=== SUMMARY ===
Isotope   Reaction  Datasets  Rows      Flagged   
Cr-50     n_g       21        5573      274       
Cr-50     n_p       1         1         0         
Cr-50     n_a       2         28        0         
Cr-53     n_g       11        26194     2747      
Cr-53     n_p       19        48        0         
Cr-53     n_a       1         7         0         


In [ ]:

# --- Cell 6: zip and download ---
zip_name = "exfor_cr_3channels.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(output_dir):
        for file in files:
            filepath = os.path.join(root, file)
            zf.write(filepath, os.path.relpath(filepath, output_dir))

print(f"\nDone. '{zip_name}' created with 6 CSV files.")

try:
    from google.colab import files
    files.download(zip_name)
except ImportError:
    print("Not in Colab -- find the zip in the file browser and download manually.")


Done. 'exfor_cr_3channels.zip' created with 6 CSV files.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>